# Подбор параметров вручную

Площадка для настройки XGBoost. Признаки, фильтр окна и кодировки взяты из решения без изменений,
меняются только параметры модели.

Файл ничего не пишет на диск и на решение не влияет. Подсчёт признаков занимает около полуминуты,
дальше один быстрый заход около семи секунд.

In [1]:
import os

# ноутбук лежит в theories/, а данные в корне проекта
if not os.path.isdir("data") and os.path.isdir(os.path.join("..", "data")):
    os.chdir("..")
print("рабочая папка:", os.getcwd())

рабочая папка: C:\Users\user\PycharmProjects\ML Avito


In [2]:
import re
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

EVENT_NAMES = ["search_results_view", "item_view", "photo_swipe", "seller_page_view",
               "contact_phone_show", "contact_chat_open", "contact_message_sent",
               "favorite_add", "login"]
CLIENTS = ["Scrapy", "curl", "node-fetch", "urllib3", "requests", "Go-http-client"]
UA_FAMILIES = ["chrome", "firefox", "safari", "yabrowser", "avito_app", "client"]
UA_OSES = ["windows", "macos", "linux_x11", "android", "ios"]
PLATFORMS = ["desktop", "web", "android", "ios"]
TE_COLS = ["item_te_mean", "item_te_max", "item_known_frac"]


# Метрика кейса. Код скопирован из metric.py организаторов без единого изменения, чтобы мои
# локальные числа считались ровно тем же, чем считает проверяющая система.

def pr_curve(y_true, score):
    """(precision, recall) в точках на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]      # последняя строка каждой группы равных score
    return tp[ends] / k[ends], tp[ends] / n_pos


def precision_at_recall(y_true, score, recall=0.70):
    """Максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0


# самопроверка копии на примерах из описания метрики
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5    # равные score идут одной группой
assert precision_at_recall([0, 0, 1, 1], [0.5] * 4) == 0.5    # порядок строк не влияет

In [3]:
def clip_to_window(events_raw, meta):
    """Единственная дверь к событиям: только то, что попало внутрь окна строки.

    Всё за правой границей окна лежит в будущем относительно момента решения. Assert ловит captcha_shown,
    которая целиком лежит после окна.
    """
    ev = events_raw.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]],
                          on="cookie_id", how="inner")
    keep = (ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])
    ev = ev.loc[keep].sort_values(["cookie_id", "event_ts"], kind="mergesort").reset_index(drop=True)
    ev["platform_norm"] = ev["platform"].astype("string").str.lower().replace({"iphone": "ios"})
    assert (ev["event_name"] == "captcha_shown").sum() == 0
    return ev


def entropy(s):
    p = s.value_counts(normalize=True)
    return float(-(p * np.log(p)).sum()) if len(p) else np.nan


def ua_parse(s):
    fam = "client" if any(c in s for c in CLIENTS) else next(
        (n for k, n in [("okhttp", "avito_app"), ("YaBrowser", "yabrowser"), ("Firefox", "firefox"),
                        ("Chrome", "chrome"), ("Safari", "safari")] if k in s), "other")
    # порядок важен: в строках айфона есть "like Mac OS X", поэтому мобильные проверяем первыми
    os_ = next((n for k, n in [("Windows NT", "windows"), ("Android", "android"),
                               ("iPhone", "ios"), ("iPad", "ios"),
                               ("Macintosh", "macos"), ("Mac OS X", "macos"),
                               ("X11", "linux_x11")] if k in s), "other")
    ver = np.nan
    for pat in [r"Chrome/(\d+)", r"Firefox/(\d+)", r"Version/(\d+)", r"Avito/(\d+)", r"Scrapy/(\d+)",
                r"curl/(\d+)", r"node-fetch/(\d+)", r"urllib3/(\d+)", r"requests/(\d+)", r"Go-http-client/(\d+)"]:
        m = re.search(pat, s)
        if m:
            ver = float(m.group(1))
            break
    return fam, os_, ver


def population_stats(events, train, test):
    """Популярность объявления и частота UA накопительно: только по прошлым дням."""
    ev = pd.concat([clip_to_window(events, train), clip_to_window(events, test)], ignore_index=True)
    pr = ev.dropna(subset=["item_id"])[["item_id", "cookie_id", "window_start_ts"]].drop_duplicates()
    day = pr.groupby(["item_id", "window_start_ts"]).size().unstack(fill_value=0).sort_index(axis=1)
    item_pop = day.cumsum(axis=1).shift(1, axis=1).fillna(0).stack()

    up = ev[["user_agent", "cookie_id", "window_start_ts"]].drop_duplicates()
    ud = up.groupby(["user_agent", "window_start_ts"]).size().unstack(fill_value=0).sort_index(axis=1)
    ub = ud.cumsum(axis=1).shift(1, axis=1).fillna(0)
    ua_freq = (ub / ub.sum(axis=0).replace(0, np.nan)).stack()

    ua = pd.Series(events["user_agent"].unique())
    parsed = [ua_parse(s) for s in ua]
    info = pd.DataFrame({"user_agent": ua.values,
                         "ua_family": [p[0] for p in parsed],
                         "ua_os": [p[1] for p in parsed],
                         "ua_ver": [p[2] for p in parsed]}).set_index("user_agent")
    info["ua_is_client"] = (info["ua_family"] == "client").astype(float)
    return {"item_pop": item_pop, "ua_freq": ua_freq, "ua_info": info}

In [4]:
def build_features(meta, events_raw, pop):
    """86 признаков поведения куки. Одна функция на train и test, чтобы наборы не разъехались."""
    ev = clip_to_window(events_raw, meta)
    idx = pd.Index(meta["cookie_id"].values, name="cookie_id")
    F = pd.DataFrame(index=idx)
    g = ev.groupby("cookie_id", sort=False)
    win_h = (meta["window_end_ts"] - meta["window_start_ts"]).dt.total_seconds().values / 3600.0
    cookie_day = pd.Series(meta["window_start_ts"].values, index=meta["cookie_id"].values)

    def put(name, val):
        F[name] = val.reindex(idx) if isinstance(val, pd.Series) else val

    # объём: сколько событий и на сколько разных объявлений их хватило
    put("n_events", g.size())
    F["n_events"] = F["n_events"].fillna(0)
    put("n_items_uniq", g["item_id"].nunique())
    put("n_cat_uniq", g["item_category"].nunique())
    put("n_loc_uniq", g["item_location"].nunique())
    put("n_query_uniq", g["search_query"].nunique())
    put("events_per_hour", F["n_events"].values / win_h)
    cnt = pd.crosstab(ev["cookie_id"], ev["event_name"]).reindex(columns=EVENT_NAMES, fill_value=0)
    cnt = cnt.reindex(idx).fillna(0.0)
    for c in EVENT_NAMES:
        put("cnt_" + c, cnt[c])

    ev = ev.copy()
    ev["dt"] = g["event_ts"].diff().dt.total_seconds()
    gd = ev.dropna(subset=["dt"]).groupby("cookie_id")["dt"]
    put("dt_median", gd.median()); put("dt_min", gd.min()); put("dt_std", gd.std())
    put("dt_frac_lt_1s", gd.apply(lambda s: (s < 1).mean()))
    put("span_seconds", (g["event_ts"].max() - g["event_ts"].min()).dt.total_seconds())
    put("n_active_hours", ev.assign(h=ev["event_ts"].dt.hour).groupby("cookie_id")["h"].nunique())

    views = F["cnt_item_view"].clip(lower=1)
    contacts = F["cnt_contact_phone_show"] + F["cnt_contact_chat_open"] + F["cnt_contact_message_sent"]
    put("photo_per_view", F["cnt_photo_swipe"] / views)
    put("contact_per_view", contacts / views)
    put("view_per_search", F["cnt_item_view"] / F["cnt_search_results_view"].clip(lower=1))
    put("max_search_page", g["search_page"].max())
    put("mean_search_page", g["search_page"].mean())
    put("ptr_frac", g["pointer_x"].apply(lambda s: s.notna().mean()))
    put("ptr_x_nunique", g["pointer_x"].nunique())
    xy = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    xy["_xy"] = xy["pointer_x"].astype(int).astype(str) + "_" + xy["pointer_y"].astype(int).astype(str)
    put("ptr_xy_nunique", xy.groupby("cookie_id")["_xy"].nunique())
    F["ptr_xy_nunique"] = F["ptr_xy_nunique"].fillna(0.0)
    put("n_platform_norm", g["platform_norm"].nunique())
    put("n_user_agent", g["user_agent"].nunique())
    put("cookie_age_days",
        (meta["window_end_ts"] - meta["cookie_created_at"]).dt.total_seconds().values / 86400.0)
    put("window_dow", meta["window_start_ts"].dt.dayofweek.values.astype(float))

    # разброс по каталогу: сидит в одной теме или ходит по всему сайту
    evc = ev.dropna(subset=["item_category"])
    put("cat_entropy", evc.groupby("cookie_id")["item_category"].apply(entropy))
    put("cat_top_frac", evc.groupby("cookie_id")["item_category"].apply(
        lambda s: s.value_counts(normalize=True).max()))
    evc = evc.copy(); evc["prev"] = evc.groupby("cookie_id")["item_category"].shift()
    put("cat_switch_frac", evc.assign(
        sw=((evc["prev"].notna()) & (evc["prev"] != evc["item_category"])).astype(float)
    ).groupby("cookie_id")["sw"].mean())
    evl = ev.dropna(subset=["item_location"])
    put("loc_entropy", evl.groupby("cookie_id")["item_location"].apply(entropy))
    put("loc_top_frac", evl.groupby("cookie_id")["item_location"].apply(
        lambda s: s.value_counts(normalize=True).max()))
    evi = ev.dropna(subset=["item_id"]); gi = evi.groupby("cookie_id")
    put("item_repeat_frac", 1 - gi["item_id"].nunique() / gi.size())
    key = pd.MultiIndex.from_arrays([evi["item_id"].values, evi["window_start_ts"].values])
    put("item_pop_mean", evi.assign(p=pop["item_pop"].reindex(key).values)
        .groupby("cookie_id")["p"].mean())
    stt = pd.crosstab(ev["cookie_id"], ev["seller_type"], normalize="index")
    put("seller_frac_private", stt["private"] if "private" in stt else pd.Series(0.0, index=stt.index))
    put("seller_known_frac", g["seller_type"].apply(lambda s: s.notna().mean()))

    # ритм: длина пауз между действиями и насколько они ровные
    put("dt_mean", gd.mean()); put("dt_max", gd.max())
    put("dt_p10", gd.quantile(0.10)); put("dt_p90", gd.quantile(0.90))
    put("dt_frac_lt_5s", gd.apply(lambda s: (s < 5).mean()))
    F["dt_cv"] = F["dt_std"] / F["dt_mean"].clip(lower=1e-9)
    ev["new_sess"] = ev["dt"].isna() | (ev["dt"] > 1800)
    ev["sess_id"] = ev.groupby("cookie_id")["new_sess"].cumsum()
    sess = ev.groupby(["cookie_id", "sess_id"]).size().rename("n").reset_index()
    put("n_sessions", sess.groupby("cookie_id")["sess_id"].max())
    put("sess_events_mean", sess.groupby("cookie_id")["n"].mean())
    put("sess_events_max", sess.groupby("cookie_id")["n"].max())
    put("hour_entropy", ev.assign(h=ev["event_ts"].dt.hour).groupby("cookie_id")["h"].apply(entropy))
    put("night_frac", ev.assign(n=(ev["event_ts"].dt.hour < 6).astype(float))
        .groupby("cookie_id")["n"].mean())

    # поиск: как глубоко листает выдачу и повторяет ли один запрос
    srch = ev[ev["event_name"] == "search_results_view"].copy(); gs = srch.groupby("cookie_id")
    put("page_gt5_frac", gs["search_page"].apply(lambda s: (s > 5).mean()))
    put("page_gt10_frac", gs["search_page"].apply(lambda s: (s > 10).mean()))
    put("page_std", gs["search_page"].std())
    srch["prev_page"] = gs["search_page"].shift()
    put("page_step_plus1_frac", srch.assign(
        st=((srch["search_page"] - srch["prev_page"]) == 1).astype(float)
    ).groupby("cookie_id")["st"].mean())
    put("q_repeat_frac", 1 - gs["search_query"].nunique() / gs.size().clip(lower=1))
    put("q_len_mean", gs["search_query"].apply(lambda s: s.dropna().str.len().mean()))

    # курсор: есть ли координаты вообще и как он по экрану ходит
    pxy = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    pxy["dist"] = np.sqrt(pxy.groupby("cookie_id")["pointer_x"].diff() ** 2
                          + pxy.groupby("cookie_id")["pointer_y"].diff() ** 2)
    pxy["_xy"] = pxy["pointer_x"].astype(int).astype(str) + "_" + pxy["pointer_y"].astype(int).astype(str)
    gp = pxy.groupby("cookie_id")
    put("ptr_x_std", gp["pointer_x"].std()); put("ptr_y_std", gp["pointer_y"].std())
    put("ptr_dist_mean", gp["dist"].mean()); put("ptr_dist_std", gp["dist"].std())
    put("ptr_repeat_frac", 1 - gp["_xy"].nunique() / gp.size())

    # устройство: браузер, ОС, платформа и редкость такого user-agent
    ua_ck = ev.groupby("cookie_id")["user_agent"].agg(lambda s: s.mode().iat[0])
    uj = pop["ua_info"].reindex(ua_ck.values)
    put("ua_is_client", pd.Series(uj["ua_is_client"].values, index=ua_ck.index))
    put("ua_ver", pd.Series(uj["ua_ver"].values, index=ua_ck.index))
    uk = pd.MultiIndex.from_arrays([ua_ck.values, cookie_day.reindex(ua_ck.index).values])
    put("ua_freq", pd.Series(pop["ua_freq"].reindex(uk).values, index=ua_ck.index))
    for fam in UA_FAMILIES:
        put("ua_fam_" + fam, pd.Series((uj["ua_family"].values == fam).astype(float), index=ua_ck.index))
    for os_ in UA_OSES:
        put("ua_os_" + os_, pd.Series((uj["ua_os"].values == os_).astype(float), index=ua_ck.index))
    pl = pd.crosstab(ev["cookie_id"], ev["platform_norm"], normalize="index")
    for p in PLATFORMS:
        put("plat_frac_" + p, pl[p] if p in pl else pd.Series(0.0, index=pl.index))
    ev["prev_plat"] = ev.groupby("cookie_id")["platform_norm"].shift()
    put("plat_switch_frac", ev.assign(
        sw=((ev["prev_plat"].notna()) & (ev["prev_plat"] != ev["platform_norm"])).astype(float)
    ).groupby("cookie_id")["sw"].mean())

    zero = ["n_events", "events_per_hour", "ptr_xy_nunique"] + ["cnt_" + c for c in EVENT_NAMES]
    F[zero] = F[zero].fillna(0.0)
    return F.reset_index(drop=True)

In [5]:
def cov_index(events, train, test):
    """Кто с кем пересекался по объявлениям, только по прошлым дням.

    Скраперы обходят каталог по спискам, и списки у разных сборщиков пересекаются. Двум живым
    людям совпасть по нескольким объявлениям почти нереально.
    """
    ev = pd.concat([clip_to_window(events, train), clip_to_window(events, test)])
    pr = ev.dropna(subset=["item_id"])[["cookie_id", "item_id", "window_start_ts"]].drop_duplicates()
    part = defaultdict(Counter)
    for _, grp in pr.groupby("item_id"):
        ck, dy = grp["cookie_id"].values, grp["window_start_ts"].values
        if len(ck) < 2 or len(ck) > 60:      # слишком популярные связывают всех со всеми
            continue
        for i in range(len(ck)):
            for j in range(len(ck)):
                if i != j and dy[j] < dy[i]:
                    part[ck[i]][ck[j]] += 1
    return part


def cov_features(meta, cov):
    idx = pd.Index(meta["cookie_id"].values)
    f = lambda d: pd.Series(d).reindex(idx).fillna(0).values
    return pd.DataFrame({
        "cov_partners": f({k: len(v) for k, v in cov.items()}),
        "cov_max_shared": f({k: max(v.values()) for k, v in cov.items()}),
        "cov_strong": f({k: sum(1 for x in v.values() if x >= 2) for k, v in cov.items()})})


def item_pairs(meta, events):
    ev = clip_to_window(events, meta).dropna(subset=["item_id"])
    p = ev[["cookie_id", "item_id"]].drop_duplicates()
    pos = pd.Series(np.arange(len(meta)), index=meta["cookie_id"].values)
    return p.assign(row=pos.reindex(p["cookie_id"]).values)[["row", "item_id"]]


def encode_items(fit_pairs, fit_y, apply_pairs, n_apply, k=10.0, drop_own=True):
    """Репутация объявления: доля ботов среди его зрителей, со сглаживанием.

    Статистика только по fit_pairs. Для обучающих строк вычитается собственный вклад, иначе
    признак предсказывал бы метку, из которой сам сделан.
    """
    p = float(np.mean(np.asarray(fit_y)[np.unique(fit_pairs["row"].values)]))
    fp = fit_pairs.assign(y=np.asarray(fit_y)[fit_pairs["row"].values])
    st = fp.groupby("item_id")["y"].agg(["sum", "size"])
    s = apply_pairs["item_id"].map(st["sum"]).fillna(0.0).values
    n = apply_pairs["item_id"].map(st["size"]).fillna(0.0).values
    if drop_own:
        s = s - np.asarray(fit_y)[apply_pairs["row"].values].astype(float)
        n = n - 1.0
    d = pd.DataFrame({"row": apply_pairs["row"].values, "enc": (s + k * p) / (n + k),
                      "known": (n > 0).astype(float)})
    g = d.groupby("row")
    out = np.full((n_apply, 3), np.nan)
    out[g["enc"].mean().index.values, 0] = g["enc"].mean().values
    out[g["enc"].max().index.values, 1] = g["enc"].max().values
    out[g["known"].mean().index.values, 2] = g["known"].mean().values
    return out

In [6]:
dates = ["cookie_created_at", "window_start_ts", "window_end_ts"]
train = pd.read_csv("data/train.csv", parse_dates=dates)
test = pd.read_csv("data/test.csv", parse_dates=dates)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])
ytr = train["target"].values.astype(int)

pop = population_stats(events, train, test)
Xtr, Xte = build_features(train, events, pop), build_features(test, events, pop)
cov = cov_index(events, train, test)
Xtr = pd.concat([Xtr, cov_features(train, cov)], axis=1)
Xte = pd.concat([Xte, cov_features(test, cov)], axis=1)
FEATURES = list(Xtr.columns)
P_tr, P_te = item_pairs(train, events), item_pairs(test, events)

print("train", train.shape, "| test", test.shape, "| ботов", ytr.sum(), f"({ytr.mean():.1%})")
print("признаков:", len(FEATURES), "плюс 3 кодировки, которые считаются внутри фолдов")

train (11091, 5) | test (4909, 4) | ботов 899 (8.1%)
признаков: 86 плюс 3 кодировки, которые считаются внутри фолдов


## Как пользоваться

Ниже те же признаки и та же кросс-валидация, что в решении, только вместо финальной модели
площадка: ставишь параметры, получаешь метрику.

`try_params` берёт параметры решения и накладывает сверху то, что передал. `try_grid` крутит одну
ручку по списку значений.

```python
try_params(max_depth=7, reg_lambda=10)
try_grid("max_depth", [4, 5, 6, 7, 8])
```

Два режима. Быстрый, по умолчанию: одно разбиение на пять фолдов, секунд семь, годится чтобы
понять направление. Полный, `full=True`: пятнадцать случайных разбиений плюс хронологические
фолды плюс bootstrap, около двух минут, и сразу вердикт.

Быстрым режимом решения не принимаются. Шум метрики 0.041, а на одном разбиении разброс ещё
больше, так что разница в сотую там не значит ничего. Вердикт выносит полный режим по правилу,
записанному до экспериментов: bootstrap за изменение минимум в 90% перевыборок, знак держится на
всех трёх повторах, хронологический протокол не падает.

Опробованное копится в таблице `LOG`.

In [8]:
import time
from sklearn.model_selection import RepeatedStratifiedKFold

# параметры текущего решения, всё сравнивается с ними
DEFAULT = dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
               colsample_bytree=0.8)
FIXED = dict(tree_method="hist", eval_metric="logloss", random_state=SEED,
             verbosity=0, n_jobs=-1)

folds_quick = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xtr, ytr))
folds_full = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3,
                                          random_state=777).split(Xtr, ytr))
day = train.window_start_ts.dt.normalize()
folds_time = [(np.where(((day >= a) & (day <= b)).values)[0],
               np.where(((day >= c) & (day <= d)).values)[0])
              for a, b, c, d in [("2026-04-06", "2026-04-13", "2026-04-14", "2026-04-15"),
                                 ("2026-04-06", "2026-04-15", "2026-04-16", "2026-04-17"),
                                 ("2026-04-06", "2026-04-17", "2026-04-18", "2026-04-19")]]


def oof(params, folds, nrep=1):
    """Предсказания вне обучения. Кодировка пересчитывается внутри каждого фолда."""
    out = np.full((nrep, len(ytr)), np.nan)
    per = len(folds) // nrep
    for k, (a, b) in enumerate(folds):
        sa, sb = set(a), set(b)
        fp, vp = P_tr[P_tr.row.isin(sa)], P_tr[P_tr.row.isin(sb)]
        A, B = Xtr.iloc[a][FEATURES].copy(), Xtr.iloc[b][FEATURES].copy()
        ea = encode_items(fp, ytr, fp, len(ytr), drop_own=True)
        eb = encode_items(fp, ytr, vp, len(ytr), drop_own=False)
        for i, c in enumerate(TE_COLS):
            A[c], B[c] = ea[a, i], eb[b, i]
        m = XGBClassifier(**params, **FIXED)
        m.fit(A, ytr[a])
        out[k // per, b] = m.predict_proba(B)[:, 1]
    return np.nanmean(out, axis=0), out


def score(p, target=0.70):
    s = ~np.isnan(p)
    return precision_at_recall(ytr[s], p[s], target)

In [9]:
LOG = []
_rng = np.random.default_rng(SEED)          # один генератор на все перевыборки
BOOT = [_rng.integers(0, len(ytr), len(ytr)) for _ in range(400)]


def try_params(name=None, full=False, **overrides):
    """Прогнать параметры решения с заменой того, что передали."""
    params = {**DEFAULT, **overrides}
    name = name or ", ".join(f"{k}={v}" for k, v in overrides.items()) or "текущие"
    t0 = time.time()
    if not full:
        p, _ = oof(params, folds_quick)
        row = dict(вариант=name, режим="быстрый", P70=score(p), P80=score(p, 0.80),
                   хроно=np.nan, условий=np.nan, сек=time.time() - t0)
        print(f"{name}: P@R70 {row['P70']:.4f}  P@R80 {row['P80']:.4f}   "
              f"(быстро, {row['сек']:.0f}с; база здесь же {BASE_QUICK:.4f})")
    else:
        p, rep = oof(params, folds_full, nrep=3)
        t, _ = oof(params, folds_time)
        mm = ~(np.isnan(P_DEF) | np.isnan(p))
        yy, aa, bb = ytr[mm], P_DEF[mm], p[mm]
        dd = np.array([precision_at_recall(yy[i], bb[i]) - precision_at_recall(yy[i], aa[i])
                       for i in BOOT])
        reps = [score(rep[i]) - score(REP_DEF[i]) for i in range(3)]
        dt = score(t) - score(T_DEF)
        ok = (dd.mean() > 0, (dd > 0).mean() >= 0.9, all(x > 0 for x in reps), dt > 0)
        row = dict(вариант=name, режим="полный", P70=score(p), P80=score(p, 0.80),
                   хроно=score(t), условий=sum(ok), сек=time.time() - t0)
        print(f"{name}: P@R70 {row['P70']:.4f}  P@R80 {row['P80']:.4f}  "
              f"хроно {row['хроно']:.4f}   ({row['сек']:.0f}с)")
        print(f"   база:  P@R70 {score(P_DEF):.4f}  P@R80 {score(P_DEF, 0.80):.4f}  "
              f"хроно {score(T_DEF):.4f}")
        print(f"   bootstrap {dd.mean():+.4f} ({(dd > 0).mean():.0%}) | "
              f"повторы {[round(x, 4) for x in reps]} | хронология {dt:+.4f} | "
              f"{sum(ok)}/4 {'ПРИНЯТО' if all(ok) else 'отклонено'}")
    row["параметры"] = dict(overrides)
    LOG.append(row)
    return row


def try_grid(param, values, full=False, **fixed):
    """Прогнать одну ручку по списку значений."""
    for v in values:
        try_params(name=f"{param}={v}", full=full, **{param: v}, **fixed)
    return pd.DataFrame(LOG)[["вариант", "режим", "P70", "P80", "хроно", "условий"]].tail(len(values))


BASE_QUICK = score(oof(DEFAULT, folds_quick)[0])
P_DEF, REP_DEF = oof(DEFAULT, folds_full, nrep=3)
T_DEF, _ = oof(DEFAULT, folds_time)
print(f"база: быстрый протокол {BASE_QUICK:.4f}, полный {score(P_DEF):.4f}, "
      f"хронология {score(T_DEF):.4f}")

база: быстрый протокол 0.8736, полный 0.8764, хронология 0.8567


In [12]:
# Площадка. Ставь параметры и запускай.

try_params()                                  # текущие, для сверки
# try_grid("max_depth", [4, 5, 6, 7, 8])
try_params(n_estimators=500, learning_rate=0.02)
# try_params("мой кандидат", full=True, max_depth=7, reg_lambda=10)

pd.DataFrame(LOG)[["вариант", "режим", "P70", "P80", "хроно", "условий", "сек"]]

текущие: P@R70 0.8736  P@R80 0.7115   (быстро, 6с; база здесь же 0.8736)
n_estimators=500, learning_rate=0.02: P@R70 0.8666  P@R80 0.7143   (быстро, 5с; база здесь же 0.8736)


,вариант,режим,P70,P80,хроно,условий,сек
0,текущие,быстрый,0.873626,0.711462,NaN,NaN,5.425802
1,learning_rate=0.1,быстрый,0.863574,0.704501,NaN,NaN,4.917929
2,текущие,быстрый,0.873626,0.711462,NaN,NaN,5.407506
3,"n_estimators=200, learning_rate=0.1",быстрый,0.864198,0.696999,NaN,NaN,2.467827
4,текущие,быстрый,0.873626,0.711462,NaN,NaN,5.741022
5,"n_estimators=500, learning_rate=0.02",быстрый,0.866575,0.714286,NaN,NaN,5.467182
